# Data Manipulation

To get anything done in machine learning, we need a way to store and manipulate data efficiently in memory. We start with $n$-dimensional arrays, which in deep learning are called *tensors*.

To see why specialized array containers are indispensable, consider a single *minibatch* of $256$ grayscale photographs of resolution $28 \times 28$ pixels, containing $256 \times 28 \times 28 = 200{,}704$ numbers. Image files store each pixel as an integer in $\{0, \ldots, 255\}$, which we rescale to $[0, 1]$ by dividing by $255$. In plain Python, three nested loops would execute $200{,}704$ separate scalar divisions in the interpreter. In a tensor library, the single expression `images / 255` dispatches to compiled C++ or CUDA kernels that divide all $200{,}704$ numbers in parallel on a CPU or GPU. Throughout deep learning, we operate on whole tensors at once rather than iterating over individual scalar elements in Python loops.

Across all modern deep learning frameworks, the tensor class (`ndarray` in MXNet, `Tensor` in PyTorch and TensorFlow, and `jax.Array` in JAX) closely mirrors NumPy's `ndarray` [@Harris.Millman.Walt.ea.2020], with two crucial additions:

1. **Hardware acceleration:** framework tensors execute across GPUs and specialized AI accelerators, whereas NumPy runs on CPUs.
2. **Automatic differentiation:** frameworks track computations on tensors and compute gradients automatically via backpropagation (that section).

We import PyTorch (`torch`).

In [ ]:
import torch

## Tensor Creation

### Shape

A tensor arranges numerical values along one or more *axes*. A tensor with one axis is a *vector*, with two axes a *matrix*, and with $k > 2$ axes a $k^\textrm{th}$-*order tensor* (its number of axes is its *order*, exposed in code as `ndim`). Our running minibatch of grayscale images is a $4^\textrm{th}$-order tensor of shape $(256, 1, 28, 28)$: axis $0$ indexes the image in the batch, axis $1$ the color channel ($1$ for grayscale, $3$ for RGB), axis $2$ the vertical pixel row, and axis $3$ the horizontal pixel column.

Calling `arange(n)` creates a 1D vector of evenly spaced values from $0$ (included) to $n$ (excluded). Newly created tensors reside in main memory for CPU computation by default; here we request 32-bit floating-point values (`float32`):

In [ ]:
x = torch.arange(12, dtype=torch.float32)
x

Each value in a tensor is an *element*. We can query the total number of elements---its *size*---and its *shape* tuple (listing the length along each axis):

In PyTorch, the total element count is returned by `numel()`.

In [ ]:
x.numel()

In [ ]:
x.shape

the figure illustrates tensors of orders $0$ through $3$. A single number (*scalar*) is a $0^\textrm{th}$-order tensor with empty shape `()`. For any tensor, the size equals the product of the lengths in its shape tuple ($256 \times 1 \times 28 \times 28 = 200{,}704$ for our image minibatch).

![Tensors with zero, one, two, and three axes. The shape lists the length of each axis, and the size is the product of these lengths: 1, 4, 12, and 24 elements.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-rank-ladder.svg)

### Data Types

Every tensor carries a *data type* (`dtype`) specifying the binary representation of each element. Deep learning models perform most arithmetic in 32-bit floating point (`float32`, $4$ bytes per element, $\approx 7$ decimal digits of precision) or 16-bit formats (`float16` / `bfloat16`, $2$ bytes per element; see that section). Raw image pixels are typically stored as 8-bit unsigned integers (`uint8`, $1$ byte per pixel for $\{0, \ldots, 255\}$), so our $200{,}704$-pixel minibatch occupies $0.2\textrm{ MB}$ as `uint8` and $0.8\textrm{ MB}$ as `float32`.

`torch.arange` defaults to `int64` on integer arguments; cast with `x.to(torch.int32)`.

### Initial Values

The functions `zeros` and `ones` construct a tensor of any shape filled with $0$s or $1$s (used for biases or buffers):

In [ ]:
torch.zeros((2, 3, 4))

Neural network weights are typically initialized by sampling elements independently from a probability distribution (that section). Below, we draw a $3 \times 4$ matrix from a standard Gaussian distribution (mean $0$, standard deviation $1$):

In [ ]:
torch.randn(3, 4)

Computer-generated random numbers are *pseudorandom*: a deterministic algorithm computes them from an integer *seed*, so resetting the seed reproduces the exact same sequence (`a` and `c` below match, whereas `b` is drawn after the generator has advanced):

PyTorch resets its global generator via `torch.manual_seed`.

In [ ]:
torch.manual_seed(42)
a = torch.randn(3)
b = torch.randn(3)  # The generator has moved on
torch.manual_seed(42)
c = torch.randn(3)  # Same seed: c repeats a
a, b, c

We can also supply exact values directly via nested Python lists:

In [ ]:
torch.tensor([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

Finally, we construct our running synthetic minibatch `images` of shape `(256, 1, 28, 28)` and type `uint8`, filled with random integers in $\{0, \ldots, 255\}$:

In [ ]:
# 256 grayscale images of 28 x 28 pixels, each pixel an integer in [0, 255]
images = torch.randint(0, 256, (256, 1, 28, 28), dtype=torch.uint8)
images.shape, images.dtype

## Reshaping and Indexing

### Reshaping

Calling `reshape` reconfigures a tensor's axes while preserving its element count and row-major order in memory (the figure). Here we reshape our 12-element vector `x` into a $3 \times 4$ matrix `X`:

![Reshaping a vector of 12 elements into a $3 \times 4$ matrix. The elements keep their order and fill the matrix one row at a time.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-reshape.svg)

In [ ]:
X = x.reshape(3, 4)
X

Because the total size is fixed, passing `-1` for one dimension lets the framework infer its length automatically—for example, flattening our $(256, 1, 28, 28)$ image minibatch into a $(256, 784)$ matrix with one image per row:

In [ ]:
images.reshape(256, -1).shape

Indexing with `None` inserts a new singleton axis of length $1$ at that position, while `squeeze` removes an axis of length $1$:

In PyTorch, `image.unsqueeze(0)` also inserts a singleton axis at position `0`.

In [ ]:
image = images[0]
image[None].shape, image.squeeze(0).shape

### Indexing and Slicing

Elements and sub-tensors are accessed via $0$-based integer indexing (negative indices count backward from the end) and half-open slices `start:stop` (including `start`, excluding `stop`), separated by commas across axes:

In [ ]:
X[-1], X[1:3]

Notice the shape difference (the figure): an integer index (`X[-1]`) collapses that axis to return a vector of shape `(4,)`, whereas a slice (`X[1:3]`) preserves the axis to return a matrix of shape `(2, 4)`.

![Indexing a $3 \times 4$ matrix. The integer index `X[-1]` selects the last row and removes axis 0, leaving shape $(4,)$. The slice `X[1:3]` selects rows 1 and 2 and keeps axis 0, leaving shape $(2, 4)$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-index-read.svg)

On our image minibatch of shape $(256, 1, 28, 28)$, `images[0]` extracts the first image $(1, 28, 28)$, `images[:32]` slices the first $32$ images $(32, 1, 28, 28)$, and `images[:, 0, 14, 14]` extracts the center pixel across all $256$ images $(256,)$:

In [ ]:
images[0].shape, images[:32].shape, images[:, 0, 14, 14].shape

We can also index a tensor using *integer index tensors* (*gather*): passing two 1D integer tensors `rows = (0, 1, 2)` and `cols = (3, 0, 2)` selects entries `(0, 3)`, `(1, 0)`, and `(2, 2)`:

In [ ]:
rows, cols = torch.arange(3), torch.tensor([3, 0, 2])
X[rows, cols]

Similarly, an elementwise comparison produces a *Boolean mask* of `True`/`False` values (combinable with `&`, `|`, `~`). Indexing `X[mask]` extracts all `True` entries as a 1D vector, whereas `where(mask, A, B)` chooses from `A` or `B` elementwise while preserving the original shape:

In [ ]:
mask = (X > 2) & (X < 8)
X[mask]

In [ ]:
torch.where(mask, X, 0)

### Writing Elements

MXNet and PyTorch tensors are *mutable* (`X[i, j] = val` overwrites memory in place), whereas TensorFlow and JAX tensors are *immutable*. Moreover, when `reshape` returns a zero-copy *view* sharing the original buffer (as in PyTorch), mutating `X` also mutates `x`:
<!-- d2l:prose id=ndarray-md-23 fw=mxnet, pytorch -->

We assign `17` to `X[1, 2]` (flat index $6$) and inspect both `X` and `x`:

In [ ]:
X[1, 2] = 17
X, x

In PyTorch, `x[6]` changed to `17.` because `reshape` returned a view sharing storage with `x`.

Slicing on the left-hand side assigns to an entire block at once:

In [ ]:
X[:2, :] = 12
X

## Operations

### Elementwise Operations

Standard arithmetic operators (`+`, `-`, `*`, `/`, `**`) and unary functions (`exp`) lift scalar operations to act independently on every corresponding element of identically shaped tensors:

In [ ]:
x = torch.tensor([1.0, 2, 4, 8])
y = torch.tensor([2.0, 2, 2, 2])
x + y, x - y, x * y, x / y, x ** y

In [ ]:
torch.exp(x)

Dividing our `uint8` minibatch `images` by `255` rescales all $200{,}704$ pixels at once and promotes the result to `float32`:

In [ ]:
scaled = images / 255
print(scaled.dtype, images[0, 0, 0, :4], scaled[0, 0, 0, :4], sep='\n')

Comparisons such as `x == y` return a Boolean tensor elementwise. Summing a tensor via `sum()` collapses its entries to a scalar; on a Boolean mask, `True` counts as $1$ and `False` as $0$, so `(x == y).sum()` counts matching entries:

In [ ]:
x == y

In [ ]:
x.sum(), (x == y).sum()

### Broadcasting

Often we combine tensors of *different* shapes, such as adding a 4-element bias vector `v` to every row of a $3 \times 4$ matrix `X`:

In [ ]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
v = torch.tensor([100.0, 200, 300, 400])
X + v

Rather than materializing physical copies, tensor libraries apply a two-step *broadcasting* rule:

1. **Align shapes from the right:** Prepend $1$s on the left of the lower-order tensor's shape tuple until both shapes have the same length. Two aligned dimensions are compatible if they are equal or if one of them is $1$.
2. **Stretch singleton axes:** Virtually replicate any length-$1$ axis to match the other tensor's length $k > 1$ without copying memory, then apply the elementwise operation.

In the figure, a $3 \times 1$ column `a` and a $1 \times 2$ row `b` both stretch to produce a $3 \times 2$ matrix of pairwise sums:

![Broadcasting stretches axes of length 1. The column `a` of shape $(3, 1)$ and the row `b` of shape $(1, 2)$ are both expanded to shape $(3, 2)$ (dashed: virtual copies) before the elementwise addition.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-broadcasting.svg)

In [ ]:
a = torch.arange(3).reshape(3, 1)
b = torch.arange(2).reshape(1, 2)
a, b

In [ ]:
a + b

If any pair of aligned dimensions differs and neither is $1$---such as `(3, 2)` and `(2, 3)`---broadcasting raises an error:

In [ ]:
try:
    torch.ones((3, 2)) + torch.ones((2, 3))
except Exception as e:
    print(e)

Thus, subtracting a $(28, 28)$ mean image from `images` of shape $(256, 1, 28, 28)$ broadcasts cleanly as $(1, 1, 28, 28)$, whereas adding a per-image vector of shape $(256,)$ to flattened images $(256, 784)$ fails unless reshaped to a column $(256, 1)$.

### Concatenation

`concatenate` (or `cat` / `concat`) joins tensors end to end along an *existing* axis (all other axes must match), whereas `stack` joins identically shaped tensors along a *new* axis:

In [ ]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
torch.cat((X, Y), dim=0), torch.cat((X, Y), dim=1)

In [ ]:
torch.stack((X, Y)).shape, torch.stack((X, Y), dim=2).shape

## Memory and Conversion

### Allocation and In-Place Updates

Standard expressions like `Y = Y + X` allocate a new buffer and rebind `Y` to the new address, as verified with Python's `id()`:

In [ ]:
before = id(Y)
Y = Y + X
id(Y) == before

For models with gigabytes of parameters, unnecessary allocations waste memory bandwidth and can leave other references pointing to stale buffers (though inside forward passes, you must not overwrite activations still needed by automatic differentiation; see that section).
<!-- d2l:prose id=ndarray-md-35 fw=mxnet, pytorch -->

In MXNet and PyTorch, slice assignment `Z[:] = X + Y` writes into an existing buffer `Z`, and augmented assignment `X += Y` updates `X` in place without even a temporary buffer:

In [ ]:
Z = torch.zeros_like(Y)
before = id(Z)
Z[:] = X + Y
id(Z) == before

In [ ]:
before = id(X)
X += Y
id(X) == before

### Conversion to Other Python Objects

Converting between framework tensors and NumPy arrays is routine at the boundary with plotting or data libraries, but frameworks differ in whether conversion shares memory or copies:

On the CPU, `X.numpy()` and `torch.from_numpy(A)` **share the exact same underlying memory buffer**, so mutating one in place immediately modifies the other!

In [ ]:
A = X.numpy()
B = torch.from_numpy(A)
type(A), type(B)

In [ ]:
A[0, 0] = 100  # Write to the NumPy array
X[0, 0], B[0, 0]

To extract a single-element ($0^\textrm{th}$-order) tensor into a Python `float` or `int` (for example, when logging the scalar loss), call `.item()` or `float()`/`int()` (which forces the CPU to synchronize with any pending GPU kernels):

In [ ]:
a = torch.tensor(3.5)
a, a.item(), float(a), int(a)

> **Tip: Exploring Framework APIs.** Whenever you encounter an unfamiliar module or function, use `dir(obj)` (or Tab completion) to list available attributes, `help(fn)` or `fn?` in Jupyter to inspect its signature and docstring (and `fn??` to view its Python source), and verify your understanding with a tiny test input whose expected output you can predict by hand.

## Summary

Tensors store inputs, parameters, activations, and gradients as $n$-dimensional arrays characterized by a `shape` and a `dtype`. Deep learning code expresses computations over whole tensors at once—rescaling all $200{,}704$ pixels of a minibatch in a single compiled operation (`images / 255`). We reconfigure axes with `reshape`, extract or modify subsets with integer indices, slices, and Boolean masks (`where`), join tensors with `concatenate` and `stack`, and combine differently shaped tensors via *broadcasting* (aligning shapes from the right and stretching singleton axes). Finally, mutable frameworks (PyTorch and MXNet) support in-place updates (`Z[:] = ...`, `X += Y`), while immutable frameworks (TensorFlow and JAX) manage memory reuse through `Variable` containers or compiler optimizations.

## Exercises

1. [code] **Inferred reshape axes.** Suppose a minibatch of $n$ grayscale images of resolution $28 \times 28$ arrives as a tensor of shape $(n, 1, 28, 28)$, and each image must be flattened into a row vector of $784$ pixel values. Verify each prediction below in code:
    1. Predict the output shape of reshaping the minibatch to `(-1, 784)` versus `(256, -1)`, first for a full minibatch of $n = 256$ images and then for the final, smaller minibatch of an epoch with $n = 100$ images. Which of the two target shapes works reliably for *any* batch size $n$?
    1. Suppose the images were accidentally loaded in 3-channel RGB format of shape $(n, 3, 28, 28)$. What happens when you reshape that tensor to `(-1, 784)`? Does `reshape(-1, 784)` catch the channel-count mistake, or does it silently scramble the batch dimension?

1. [code] **Broadcasting shapes and silent bugs.** Broadcasting aligns two shape tuples from the right.
    1. Construct two $3^\textrm{rd}$-order tensors `a` and `b` whose shapes differ along every single axis yet are broadcast-compatible. Predict the shape of `a + b` by hand, then run the code to verify your prediction.
    1. A regression model outputs a column of predictions `y_hat` of shape $(5, 1)$ for five examples whose ground-truth targets `y` are stored as a 1D vector of shape $(5,)$. Predict the shape of the residual `y_hat - y`. Next, let `y` hold `[0.0, 1.0, 2.0, 3.0, 4.0]` and set `y_hat = y.reshape(5, 1)` so that every prediction equals its true label identically. Compute the mean of the squared entries of `y_hat - y`, explain why this "mean squared error" is nonzero even though the predictions are perfect, and show how to reshape `y` (or `y_hat`) so that each prediction is compared only against its own label.

1. [code] **Arithmetic limits of data types.** Raw pixels in `images` are stored as 8-bit unsigned integers (`uint8`), whereas neural network layers compute in floating point (`float32`).
    1. Predict the numerical result of adding `100` to a `uint8` tensor entry holding the value `200`, and verify your prediction in code. Explain why this wrap-around occurs, and show how to brighten the first image of `images` by $100$ intensity levels (clamped or promoted) without integer overflow.
    1. An IEEE 754 `float32` number allocates $24$ bits of significand (mantissa) precision. Explain why `float32` can represent every consecutive integer up to $2^{24} = 16{,}777{,}216$ exactly, and verify in code that evaluating `float32(2**24) + float32(1)` leaves the number unchanged at $16{,}777{,}216$. If a training loop accumulates the total number of examples or tokens processed in a `float32` scalar counter (incrementing by $1$ each time), after how many examples will the counter freeze completely, and what data type should be used instead?

4. [code] **In-place composition.** ● Evaluate `(A + B) * (-A / 2)` for two identically shaped `float32` tensors `A` and `B` *without allocating any intermediate tensor*, using only PyTorch's in-place methods (`add_`, `div_`, `neg_`, `mul_`) or `out=` keyword arguments. Confirm that `A.data_ptr()` and `B.data_ptr()` remain unchanged and that the final result matches the out-of-place expression `(A + B) * (-A / 2)`. Explain why writing `A[:] = A + B` would *not* satisfy the zero-temporary-allocation requirement.

    *Adapted from Nicolas P. Rougier, [100 numpy exercises](https://github.com/rougier/numpy-100/blob/master/100_Numpy_exercises.md), problem 35.*

5. [code] **Python loops versus broadcasting.** ● Given a 1D vector `a` of $200$ elements and a 1D vector `b` of $300$ elements, we want to compute the $200 \times 300$ pairwise absolute-difference matrix $D$ with entries $d_{ij} = |a_i - b_j|$.
    1. Compute $D$ first using two nested Python `for` loops, and then using a single broadcasted subtraction after reshaping `a` to `(200, 1)` and `b` to `(1, 300)`. Verify that both matrices agree numerically, measure the wall-clock execution time of each method, and report the speedup factor.
    1. Even though broadcasting avoids copying the inputs `a` and `b`, the output matrix $D$ must still be materialized in memory. How many megabytes does $D$ occupy in `float32` if `a` and `b` each contain $10{,}000$ elements? What lesson does this teach about the peak memory footprint of broadcasted outer operations?

[Discussions](https://d2l.discourse.group/t/27)